# 05 — Model Training & Evaluation
**Movie Review Sentiment Analysis Project**

Stages 7 & 8: Model Development and Evaluation.

We train three models and compare them on the validation set, then
evaluate the best model on the official held-out test set.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, ConfusionMatrixDisplay
)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)
from src.data_loader import load_imdb_dataset, get_train_test
from src.preprocessing import apply_cleaning

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

df = load_imdb_dataset(PROJECT_ROOT)
train_df, test_df = get_train_test(df)
train_df['clean_review'] = apply_cleaning(train_df['review'])
test_df['clean_review']  = apply_cleaning(test_df['review'])

X_train_full = train_df['clean_review'].values
y_train_full = train_df['label'].values
X_test = test_df['clean_review'].values
y_test = test_df['label'].values

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2, random_state=RANDOM_STATE, stratify=y_train_full
)
print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

## Stage 7 — Model Training
Three candidate models with TF-IDF features:

In [ ]:
TFIDF_PARAMS = dict(ngram_range=(1,2), sublinear_tf=True, min_df=2, max_features=100_000)

candidates = {
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
        ('clf',   MultinomialNB(alpha=0.1)),
    ]),
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
        ('clf',   LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=RANDOM_STATE)),
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
        ('clf',   LinearSVC(C=1.0, max_iter=2000, random_state=RANDOM_STATE)),
    ]),
}

results = []
fitted  = {}
for name, pipe in candidates.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    f1  = f1_score(y_val, y_pred, average='macro')
    results.append({'Model': name, 'Val Accuracy': acc, 'Val F1 (macro)': f1})
    fitted[name] = pipe
    print(f'{name:25s}  acc={acc:.4f}  f1={f1:.4f}')

results_df = pd.DataFrame(results).sort_values('Val F1 (macro)', ascending=False)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
results_df.plot.bar(x='Model', y=['Val Accuracy','Val F1 (macro)'], ax=ax, rot=0)
ax.set_title('Model Comparison on Validation Set')
ax.set_ylabel('Score')
ax.set_ylim(0.8, 1.0)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Stage 8 — Final Evaluation on Test Set
The best model (by validation F1) is evaluated on the official test set.

In [ ]:
best_name = results_df.iloc[0]['Model']
best_pipe = fitted[best_name]
print(f'Best model: {best_name}')

y_pred_test = best_pipe.predict(X_test)

print(f'Test Accuracy : {accuracy_score(y_test, y_pred_test):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_test, target_names=['Negative','Positive'], digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred_test)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'], ax=ax)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title(f'Confusion Matrix — {best_name}\n(Official Test Set, n=25,000)')
plt.tight_layout()
plt.show()

In [ ]:
# Save the best pipeline
MODEL_PATH = os.path.join(PROJECT_ROOT, 'models', 'sentiment_pipeline.joblib')
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
joblib.dump(best_pipe, MODEL_PATH)
print(f'Pipeline saved to: {MODEL_PATH}')